# Finder: watch development notebook

For the `jupyter_micropython_kernel` ("MicroPython - USB"). Install the kernel once
with `python3 -m jupyter_micropython_kernel.install` (after `pip install -r requirements.txt`).
Every cell after **Connect** runs **on the watch** (stock MicroPython v1.29, `ESP32_GENERIC-SPIRAM`).
Run the cells top to bottom once; after that each section can be re-run on its own.

Sections: connect, deploy, board bring-up, display benchmark, radio ping-pong,
IMU live view, run the game, live-tune thresholds.

## 1. Connect
Find the port with `ls /dev/cu.usbserial-*` (macOS). One kernel per watch.

In [ ]:
%serialconnect --port /dev/cu.usbserial-XXXX --baud 115200

## 2. Deploy (host terminal, not this kernel)

Run `%disconnect` in this kernel first (only one program can own the serial port),
then come back to **1. Connect** when the copy is done.

```sh
python3 tools/deploy.py --port /dev/cu.usbserial-XXXX --noapp   # copy code + create /noapp
python3 tools/deploy.py -n                                        # dry run: what gets copied
```

`--noapp` makes `main.py` skip the game on boot so the REPL stays free for this
notebook (`--app` removes the flag again; a double press of the side key during
the first second after boot does the same once). `deploy.py` copies `boot.py`,
`main.py`, `app/`, `finder/`, `hal/`, `ui/` (and `bma423conf.bin` if present).

`boot.py` is deliberately minimal (no drivers, no network); `main.py` does the safe-boot
check and then starts the game (`app.run`). Never flash firmware from here.

## 3. Board bring-up
`Board()` only sets the CPU clock; every part is lazy. `init(strict=False)` records failing parts in `errors` instead of raising.

In [ ]:
import gc, time
from hal.board import Board

b = Board()
b.init(strict=False)
print("errors:", b.errors)
print("pmu:", b.pmu.status())
print("imu chip id: 0x%02x" % b.imu.chip_id())   # 0x13
# Board() soft-resets the BMA423; only load_config() (or the game) loads the blob.
# True None if bma423conf.bin is on the watch and accepted.
print("imu features:", b.imu.load_config(), b.imu.feat_error)
# load_config() switches no feature on; poll_features() does (step counter,
# activity, wrist-wear), as the game does. 2 = FEAT_OK.
print("imu feature engine:", b.imu.poll_features())
# walk ~20 steps, then: print(b.imu.steps(), b.imu.activity())   # about 20; activity 1 = walk
print("touch:", b.touch.read())
print("radio mac:", b.radio.mac, "channel", b.radio.channel)
gc.collect(); print("mem_free", gc.mem_free())

In [ ]:
# quick hardware poke: backlight ramp + one motor pulse
d = b.display
for lvl in (0.2, 0.6, 1.0, 0.6):
    d.brightness(lvl); time.sleep_ms(250)
b.haptics.set(1); time.sleep_ms(80); b.haptics.set(0)

## 4. Display benchmark
Raw strip pushes, then the real renderer (`ui.renderer`). For the full breakdown (SPI clock, palette blit, gc) run `tools/bench_display.py` section by section.

In [ ]:
buf = bytearray(240 * 24 * 2)
n = 20
t0 = time.ticks_us()
for _ in range(n):
    for s in range(10):
        d.push_strip(s * 24, 24, buf)
dt = time.ticks_diff(time.ticks_us(), t0) / 1000 / n
print("10 x push_strip: %.1f ms/frame (%.1f fps max)" % (dt, 1000 / dt if dt else 0))

In [ ]:
from ui.renderer import Renderer
from finder.render_params import make_params

r = Renderer()
p = make_params(screen="WARM", zone=2, ramp="green", intensity=0.6, speed_px_s=80.0,
                pulse_period_ms=1000, glow_r_px=44.0, ring_live=True, glyph="glow",
                heartbeat="DOUBLE", status=(80, 80, 4, False, False))
gc.collect()
m0 = gc.mem_free()
n = 60
t0 = time.ticks_ms()
for _ in range(n):
    r.frame(p, d, time.ticks_ms())
dt = time.ticks_diff(time.ticks_ms(), t0) / n
print("renderer: %.1f ms/frame = %.1f fps, alloc %d B/frame" % (dt, 1000 / dt if dt else 0, (m0 - gc.mem_free()) // n))
t0 = time.ticks_us(); gc.collect(); print("gc.collect %.1f ms" % (time.ticks_diff(time.ticks_us(), t0) / 1000))

## 5. Radio ping-pong

Needs two watches (one kernel each) and `tools/radio_pingpong.py` on both
(`mpremote cp tools/radio_pingpong.py :`). Start `pong` first. Both must use the same channel.

In [ ]:
import radio_pingpong as pp
pp.run('pong', radio=b.radio)                       # watch A
# pp.run('ping', n=500, render=True, radio=b.radio) # watch B: delivery %, RTT, RSSI

In [ ]:
# passive sniff for 3 s: who is beaconing, and how loud
from finder import proto
seen = {}
def cb(mac, buf, n, rssi, t):
    if proto.valid(buf, n):
        k = bytes(mac)
        c, lo, hi = seen.get(k, (0, 0, -128))
        seen[k] = (c + 1, min(lo, rssi), max(hi, rssi))
t_end = time.ticks_add(time.ticks_ms(), 3000)
while time.ticks_diff(t_end, time.ticks_ms()) > 0:
    b.radio.poll(time.ticks_ms(), callback=cb)
    time.sleep_ms(10)
for k in seen:
    print(k, "packets %d  rssi %d..%d dBm" % seen[k])
print(b.radio.stats())

## 6. IMU live view

BMA423 FIFO (100 Hz) → `app.imu_feed.ImuFeed` → `finder.motion.MotionTracker` at 50 Hz here
(the game runtime uses `out_hz=25`: same steps/stillness, half the float work).
Shows steps, activity, stillness, tilt, face-up and bump spikes (`T.BUMP_SPIKE_G`, at most
`T.BUMP_SPIKE_MS[1]` ms), sampling at `T.BUMP_ODR_HZ` (`set_fast(True)`) as the game does while
a bump can count. Knock the watch on the table to see a tap; shake it to see a rejected (too
wide) spike. If `face_up` is False with the screen facing up, fix `BMA423_Z_SIGN` in
`hal/pins.py` (then redeploy).

In [ ]:
from app.imu_feed import ImuFeed

ACT = ("?", "still", "walk", "run")
feed = ImuFeed(b.imu)
feed.set_fast(True)                   # the game's bump rate (flushes the FIFO)
t_end = time.ticks_add(time.ticks_ms(), 15000)
t_pr = 0
while time.ticks_diff(t_end, time.ticks_ms()) > 0:
    now = time.ticks_ms()
    feed.poll(now)
    if feed.tap is not None:
        print("  TAP at", feed.tap, "peak %d mg" % feed.peak_mg)
    if time.ticks_diff(now, t_pr) >= 500:
        t_pr = now
        mt = feed.tracker
        print("steps %3d  %-5s  still %d  sd %.3f g  tilt %5.1f  face_up %d  taps %d rej %d" % (
            mt.steps, ACT[mt.activity], mt.is_still, mt.mag_sd_g,
            mt.tilt_deg, mt.face_up, feed.n_taps, feed.n_rejected))
    time.sleep_ms(20)
feed.set_fast(False)                  # back to 100 Hz before the game runs

## 7. Run the game

The same loop `main.py` runs, minus safe boot. Pair two watches (both show the same
runes, press or tap to confirm on both). Interrupt the kernel (or wait for `max_ms`)
to stop; the stats show fps and ms per stage.

In [ ]:
from app.runtime import Runtime
from app.telemetry import Telemetry

rt = Runtime(b, telemetry=Telemetry(dev="A", sid="dev1"))
try:
    rt.run(max_ms=120000)
except KeyboardInterrupt:
    pass
rt.print_stats()
print(rt.game.mode, rt.game.pair.sub, "zone", rt.game.px.zone, "buzz", rt.game.buzz)

In [ ]:
# telemetry: newest records as JSON Lines (save as .jsonl for offline analysis; there is no replay tool yet)
rt.tele.dump(20)
# rt.tele.save("/log_dev1_A.jsonl")   # or to flash

## 8. Live-tune thresholds

Change a value, then re-run `rt.run(...)`: the runtime keeps its state. `finder.tuning`
values read as `T.NAME` in the game (wrist-down time, bump window, idle dim...) and the
module constants of `finder.motion` take effect at once; tracker/feed fields are per instance.
Put what works back into `docs/design/tokens.json` (then `tools/gen_tuning.py`), not here.

In [ ]:
import finder.tuning as T
import finder.motion as M

rt.feed.thr2 = int(1.8 * 1000) ** 2   # bump spike threshold (g); default 1.5
rt.feed.spike_max_ms = 8              # widest accepted spike; default 6
T.WRIST_DOWN_MS = 3000                # on battery, screen-off delay once lowered; default 10000
T.BUMP_WINDOW_MS = 500                # both taps within this; default 400
M.STILL_ON_G = 0.015                  # |a| sd to call the wrist still; default 0.012
rt.gestures.long_ms = 700             # touch long-press (MENU); default 800
try:
    rt.run(max_ms=60000)
except KeyboardInterrupt:
    pass
rt.print_stats()

In [ ]:
# tidy up: motor off, panel asleep (LDO2 stays on: it also powers the panel)
rt.quiet()
b.display.sleep()